## <center>CSE 546: Reinforcement Learning</center>
### <center>Prof. Alina Vereshchaka</center>
#### <center>Spring 2025</center>

Welcome to the Assignment 3, Part 1: Introduction to Actor-Critic Methods! It includes the implementation of simple actor and critic networks and best practices used in modern Actor-Critic algorithms.

## Section 0: Setup and Imports

In [43]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import deque

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## Section 1: Actor-Critic Network Architectures and Loss Computation

In this section, you will explore two common architectural designs for Actor-Critic methods and implement their corresponding loss functions using dummy tensors. These architectures are:
- A. Completely separate actor and critic networks
- B. A shared network with two output heads

Both designs are widely used in practice. Shared networks are often more efficient and generalize better, while separate networks offer more control and flexibility.

---


### Task 1a – Separate Actor and Critic Networks with Loss Function

Define a class `SeparateActorCritic`. Your goal is to:
- Create two completely independent neural networks: one for the actor and one for the critic.
- The actor should output a probability distribution over discrete actions (use `nn.Softmax`).
- The critic should output a single scalar value.

 Use `nn.ReLU()` as your activation function. Include at least one hidden layer of reasonable width (e.g. 64 or 128 units).

```python
# TODO: Define SeparateActorCritic class
```

 Next, simulate training using dummy tensors:
1. Generate dummy tensors for log-probabilities, returns, estimated values, and entropies.
2. Compute the actor loss using the advantage (return - value).
3. Compute the critic loss as mean squared error between values and returns.
4. Use a single optimizer for both the Actor and the Critic. In this case, combine the actor and critic losses into a total loss and perform backpropagation.
5. Use a separate optimizers for both the Actor and the Critic. In this case, keep the actor and critic losses separate and perform backpropagation.

```python
# TODO: Simulate loss computation and backpropagation
```

🔗 Helpful references:
- PyTorch Softmax: https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html
- PyTorch MSE Loss: https://pytorch.org/docs/stable/generated/torch.nn.functional.mse_loss.html

---

In [44]:
# TODO: Define a class SeparateActorCritic with separate networks for actor and critic

# BEGIN_YOUR_CODE
class SeparateActorCritic(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(SeparateActorCritic, self).__init__()
        self.actor_net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim),
            nn.Softmax(dim=-1)
        )
        self.critic_net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        action_probs = self.actor_net(x)
        state_value = self.critic_net(x)
        return action_probs, state_value

input_dim, output_dim = 4, 2
model = SeparateActorCritic(input_dim, output_dim)

dummy_obs = torch.randn((3, input_dim))
action_probs, state_values = model(dummy_obs)
dummy_returns = torch.tensor([[1.0], [0.5], [1.5]])
dummy_log_probs = torch.log(action_probs + 1e-8)
dummy_actions = torch.tensor([0, 1, 0])
selected_log_probs = dummy_log_probs[range(3), dummy_actions]

advantages = dummy_returns - state_values.detach()
actor_loss = -(selected_log_probs * advantages.squeeze()).mean()
critic_loss = F.mse_loss(state_values, dummy_returns)
total_loss = actor_loss + critic_loss

optimizer = optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad()
total_loss.backward()
optimizer.step()

print("Actor Loss:", actor_loss.item())
print("Critic Loss:", critic_loss.item())
# END_YOUR_CODE



Actor Loss: 1.2434190511703491
Critic Loss: 2.1385695934295654


### Discuss the motivation behind each setup and when it may be preferred in practice.

YOUR ANSWER:

In [ ]:
Motivation behind completely separate actor & critic networks:
- Independence of representations: actor (policy) & critic (value func) can learn their own feature extractors without interfering with each other’s gradients.
- Architectural flexibility: can choose diff depths, widths, or even layer types for each network.
- Use case: preferred when policy & value tasks have very diff characteristics, or when suspect -ve transfer if they shared layers.

### Task 1b – Shared Network with Actor and Critic Heads + Loss Function

Now define a class `SharedActorCritic`:
- Build a shared base network (e.g., linear layer + ReLU)
- Create two heads: one for actor (output action probabilities) and one for critic (output state value)

```python
# TODO: Define SharedActorCritic class
```

Then:
1. Pass a dummy input tensor through the model to obtain action probabilities and value.
2. Simulate dummy rewards and compute advantage.
3. Compute the actor and critic losses, combine them, and backpropagate.

```python
# TODO: Simulate shared network loss computation and backpropagation
```

 Use `nn.Softmax` for actor output and `nn.Linear` for scalar critic output.

🔗 More reading:
- Policy Gradient Methods: https://spinningup.openai.com/en/latest/algorithms/vpg.html
- Actor-Critic Overview: https://www.tensorflow.org/agents/tutorials/6_reinforce_tutorial
- PyTorch Categorical Distribution: https://pytorch.org/docs/stable/distributions.html#categorical

---

In [45]:
# BEGIN_YOUR_CODE
class SharedActorCritic(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(SharedActorCritic, self).__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU()
        )
        self.actor_head = nn.Sequential(
            nn.Linear(128, output_dim),
            nn.Softmax(dim=-1)
        )
        self.critic_head = nn.Linear(128, 1)

    def forward(self, x):
        shared_out = self.shared(x)
        action_probs = self.actor_head(shared_out)
        value = self.critic_head(shared_out)
        return action_probs, value

model = SharedActorCritic(input_dim, output_dim)
dummy_obs = torch.randn((3, input_dim))
action_probs, values = model(dummy_obs)

dummy_returns = torch.tensor([[1.0], [0.5], [1.5]])
dummy_log_probs = torch.log(action_probs + 1e-8)
dummy_actions = torch.tensor([0, 1, 0])
selected_log_probs = dummy_log_probs[range(3), dummy_actions]

advantages = dummy_returns - values.detach()
actor_loss = -(selected_log_probs * advantages.squeeze()).mean()
critic_loss = F.mse_loss(values, dummy_returns)
total_loss = actor_loss + critic_loss

optimizer = optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad()
total_loss.backward()
optimizer.step()

print("Actor Loss:", actor_loss.item())
print("Critic Loss:", critic_loss.item())
# END_YOUR_CODE

Actor Loss: 0.7708703875541687
Critic Loss: 1.8106050491333008


### Discuss the motivation behind each setup and when it may be preferred in practice.

YOUR ANSWER:

In [ ]:
Motivation behind a shared actor‐critic network:
 - Parameter sharing: single “backbone” learns features useful for both policy & value, drastically reducing total parameters.
 - Improved sample efficiency: shared gradients act as a form of multi task regularization, often speeding up convergence when actor & critic tasks are related.
 - Use case: preferred in resource constrained settings, or else when know representations for policy & value estimation should overlap closely.


## Section 2: Auto-Adaptive Network Setup for Environments

You will now create a function that builds a shared actor-critic network that adapts to any Gymnasium environment. This function should inspect the environment and build input/output layers accordingly.

### Task 2: Auto-generate Input and Output Layers
Write a function `create_shared_network(env)` that constructs a neural network using the following rules:
- The input layer should match the environment's observation space.
- The output layer for the **actor** should depend on the action space:
  - For discrete actions: output probabilities using `nn.Softmax`.
  - For continuous actions: output mean and log std for a Gaussian distribution.
- The **critic** always outputs a single scalar value.

```python
# TODO: Define function `create_shared_network(env)`
```

#### Environments to Support:
Test your function with the following environments:
1. `CliffWalking-v0` (Use one-hot encoding for discrete integer observations.)
2. `LunarLander-v3` (Standard Box space for observations and discrete actions.)
3. `PongNoFrameskip-v4` (Use gym wrappers for Atari image preprocessing.)
4. `HalfCheetah-v5` (Continuous observation and continuous action.)

```python
# TODO: Loop through environments and test `create_shared_network`
```

Hint: Use `gym.spaces` utilities to determine observation/action types dynamically.

🔗 Observation/Action Space Docs:
- https://gymnasium.farama.org/api/spaces/

---

In [46]:
# BEGIN_YOUR_CODE
def create_shared_network(env):
    observation_space = env.observation_space
    act_space = env.action_space

    if isinstance(observation_space, gym.spaces.Discrete):
        observation_dim = observation_space.n
        observation_type = 'discrete'
    elif isinstance(observation_space, gym.spaces.Box):
        observation_dim = int(np.prod(observation_space.shape))
        observation_type = 'box'
    else:
        raise NotImplementedError("Unsupported observation space")

    if isinstance(act_space, gym.spaces.Discrete):
        action_dim = act_space.n
    elif isinstance(act_space, gym.spaces.Box):
        action_dim = int(np.prod(act_space.shape))
    elif isinstance(act_space, gym.spaces.MultiDiscrete):
        action_dim = int(np.sum(act_space.nvec))
    else:
        raise NotImplementedError("Unsupported action space")

    model = SharedActorCritic(observation_dim, action_dim)
    return model, observation_type
# END_YOUR_CODE


In [47]:
import ale_py
gym.register_envs(ale_py)
!pip install swig
!pip install "gymnasium[box2d]"
!pip install "gymnasium[mujoco]"

In [48]:
from gymnasium.wrappers import AtariPreprocessing
import torch.nn.functional as F

def test_create_shared_network():
    env_ids = {
        "CliffWalking-v0": lambda: gym.make("CliffWalking-v0"),
        "LunarLander-v3": lambda: gym.make("LunarLander-v3"),
        "PongNoFrameskip-v4": lambda: AtariPreprocessing(gym.make("PongNoFrameskip-v4")),
        "HalfCheetah-v5": lambda: gym.make("HalfCheetah-v5")
    }

    for name, make_env in env_ids.items():
        try:
            env = make_env()
            model, obs_type = create_shared_network(env)
            obs, _ = env.reset()

            if obs_type == 'discrete':
                obs_tensor = F.one_hot(torch.tensor(obs), num_classes=env.observation_space.n).float().unsqueeze(0)
            else:
                obs_tensor = torch.tensor(obs, dtype=torch.float32).flatten().unsqueeze(0)

            with torch.no_grad():
                action_probs, value = model(obs_tensor)
            print(f"{name}: PASS — Model forward successful. Action shape: {action_probs.shape}, Value shape: {value.shape}")
        except Exception as e:
            print(f"{name}: FAIL — {str(e)}")

test_create_shared_network()


CliffWalking-v0: PASS — Model forward successful. Action shape: torch.Size([1, 4]), Value shape: torch.Size([1, 1])
LunarLander-v3: PASS — Model forward successful. Action shape: torch.Size([1, 4]), Value shape: torch.Size([1, 1])
PongNoFrameskip-v4: PASS — Model forward successful. Action shape: torch.Size([1, 6]), Value shape: torch.Size([1, 1])
HalfCheetah-v5: PASS — Model forward successful. Action shape: torch.Size([1, 6]), Value shape: torch.Size([1, 1])


### Discuss the motivation behind each setup and when it may be preferred in practice.

YOUR ANSWER:

In [ ]:
 Section 2: Auto‐Adaptive Network Setup for Environments
 - By inspecting env obs & action spaces at runtime, we can automatically construct shared actor‑critic network that just fits any Gym env.
 - Avoids hard‑coding separate architectures per task while improving reusability, also makes agent library truly general purpose.
 - referred when planning to evaluate on many environments.

### Task 3: Write Observation Normalization Function
Create a function `normalize_observation(obs, env)` that:
- Checks if the observation space is `Box` and has `low` and `high` attributes.
- If so, normalize the input observation.
- Otherwise, return the observation unchanged.

```python
# TODO: Define `normalize_observation(obs, env)`
```

Test this function with observations from:
- `LunarLander-v3`
- `PongNoFrameskip-v4`

Note: Atari observations are image arrays. Normalize pixel values to [0, 1]. For LunarLander-v3, the different elements in the observation vector have different ranges. Normalize them to [0, 1] using the `low` and `high` attributes of the observation space.


---

In [49]:
# BEGIN_YOUR_CODE
def normalize_observation(obs, env):
    if isinstance(env.observation_space, gym.spaces.Box):
        low = env.observation_space.low
        high = env.observation_space.high
        if np.all(np.isfinite(low)) and np.all(np.isfinite(high)):
            return 2.0 * (obs - low) / (high - low) - 1.0
        elif np.issubdtype(obs.dtype, np.uint8):
            return obs.astype(np.float32) / 255.0
    return obs
# END_YOUR_CODE

In [50]:
env1 = gym.make("LunarLander-v3")
obs1, _ = env1.reset()
norm_obs1 = normalize_observation(obs1, env1)
print("Normalized LunarLander Obs:", norm_obs1)

env2 = gym.make("PongNoFrameskip-v4", render_mode="rgb_array")
obs2, _ = env2.reset()
norm_obs2 = normalize_observation(obs2, env2)
print("Normalized Pong Obs Shape:", norm_obs2.shape)

Normalized LunarLander Obs: [ 0.00239563  0.56472874  0.06066155  0.00400698 -0.00110346 -0.01374072
 -1.         -1.        ]
Normalized Pong Obs Shape: (210, 160, 3)


### Discuss the motivation behind each setup and when it may be preferred in practice.

YOUR ANSWER:

In [ ]:
 - Neural nets train more stably when inputs share a common scale.
 - Box spaces: mapping each feature to [–1, 1] centers & bounds data, speeding convergence when observation dimensions have diff numeric ranges.
 - Atari image obs: scaling uint8 pixels to [0, 1] prevents large raw pixel values from destabilizing early convolution layers.
 - Use case: normalize whenever envs expose heterogeneous or high dynamic range inputs.

## Section 4: Gradient Clipping

To prevent exploding gradients, it's common practice to clip gradients before optimizer updates.

### Task 4: Clip Gradients for Actor-Critic Networks
Use dummy tensors and apply gradient clipping with the following PyTorch method:
```python
# During training, after loss.backward():
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
```

Reuse the loss computation from Task 1a or 1b. After computing the gradients, apply gradient clipping.
Print the gradient norm before and after clipping to verify it’s applied.

🔗 PyTorch Docs: https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html


---

In [51]:
def clip_gradients(model, max_norm=0.5):
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item()**2
    total_norm = total_norm**0.5
    print(f"Gradient norm before clipping: {total_norm:.4f}")

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

    total_norm_after = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm_after += p.grad.data.norm(2).item()**2
    total_norm_after = total_norm_after**0.5
    print(f"Gradient norm after clipping:  {total_norm_after:.4f}")

input_dim, output_dim = 4, 2
model = SeparateActorCritic(input_dim, output_dim)

obs = torch.randn((3, input_dim))
probs, values = model(obs)
returns     = torch.tensor([[1.0], [0.5], [1.5]])
logp        = torch.log(probs + 1e-8)
actions     = torch.tensor([0, 1, 0])
sel_logp    = logp[range(3), actions]
adv         = returns - values.detach()

actor_loss  = -(sel_logp * adv.squeeze()).mean()
critic_loss = F.mse_loss(values, returns)
total_loss  = actor_loss + critic_loss

optimizer = optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad()
total_loss.backward()

clip_gradients(model)
optimizer.step()

Gradient norm before clipping: 11.8493
Gradient norm after clipping:  0.5000


### Discuss the motivation behind each setup and when it may be preferred in practice.

YOUR ANSWER:

In [ ]:
- Clip gradients to prevent “exploding” updates in high‑variance actor‑critic training.
- Printing norms before/after confirms the operation & helps debug training stability.
- Preferred in deep or long‑horizon RL tasks, or else whenever you observe diverging or noisy learning curves.

Contribution :   
Aditi Sinha : 50%
Suman Saurav : 50%   

If you are working in a team, provide a contribution summary.
| Team Member | Step# | Contribution (%) |
|---|---|---|
|   | Task 1 |   |
|   | Task 2 |   |
|   | Task 3 |   |
|   | Task 4 |   |
|   | **Total** |   |
